# 🛠️ 03 - Feature Engineering & Preprocessing Pipeline
**Project**: House Price Prediction & Property Analytics System  
**Objective**: Construct domain-specific features, prevent target leakage, and configure Scikit-Learn `ColumnTransformer` pipelines.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df_train = pd.read_csv('../data/processed/train.csv')
df_test = pd.read_csv('../data/processed/test.csv')


### 1. Domain Feature Construction
- **Total_Rooms**: `Bedrooms + Bathrooms`
- **Area_per_Bedroom**: `Area_sqft / max(Bedrooms, 1)`
- **Bathroom_to_Bedroom_Ratio**: `Bathrooms / max(Bedrooms, 1)`
- **Luxury_Score**: `1.5*Amenities + 1.2*Parking + 1.0*Balconies`
- **Age_Category**: Architectural lifecycle tier


In [ ]:
def engineer_features(df):
    data = df.copy()
    data['Total_Rooms'] = data['Bedrooms'] + data['Bathrooms']
    safe_bed = np.maximum(data['Bedrooms'], 1)
    data['Area_per_Bedroom'] = (data['Area_sqft'] / safe_bed).round(2)
    data['Bathroom_to_Bedroom_Ratio'] = (data['Bathrooms'] / safe_bed).round(3)
    data['Luxury_Score'] = (data['Amenities_Count'] * 1.5 + data['Parking_Spaces'] * 1.2 + data['Balconies'] * 1.0).round(2)
    
    def bin_age(a):
        if a <= 5: return 'New Construction (0-5 yrs)'
        elif a <= 15: return 'Modern (6-15 yrs)'
        elif a <= 25: return 'Mature (16-25 yrs)'
        else: return 'Established (>25 yrs)'
    data['Age_Category'] = data['Property_Age'].apply(bin_age)
    return data

train_feat = engineer_features(df_train)
test_feat = engineer_features(df_test)
train_feat[['Total_Rooms', 'Area_per_Bedroom', 'Bathroom_to_Bedroom_Ratio', 'Luxury_Score', 'Age_Category']].head()


### 2. Scikit-Learn Pipeline Assembly
Assemble numeric and categorical pipelines with `ColumnTransformer`:


In [ ]:
num_cols = [
    'Area_sqft', 'Bedrooms', 'Bathrooms', 'Parking_Spaces', 'Floors',
    'Property_Age', 'Balconies', 'Amenities_Count', 'Nearby_Schools',
    'Total_Rooms', 'Area_per_Bedroom', 'Bathroom_to_Bedroom_Ratio', 'Luxury_Score'
]
cat_cols = ['Location', 'Property_Type', 'Furnishing_Status', 'Availability', 'Age_Category']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ]
)

X_train = train_feat[num_cols + cat_cols]
X_train_trans = preprocessor.fit_transform(X_train)
print(f"Transformed Training Feature Matrix Shape: {X_train_trans.shape}")
